<a href="https://colab.research.google.com/github/hanidew/G12_IR_Evaluation/blob/main/IRWS_G12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Data Cleaning

50 topics each topic got 1000 documents

In [8]:
# --- STEP 1: MOUNT GOOGLE DRIVE ---
# This connects your Google Colab runtime to your actual Google Drive.
# When you run this, a popup will ask you to authorize Colab to access your files.
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import glob
import os

# --- STEP 2: LOAD THE QRELS (ANSWER KEY) ---
# The Qrels file tells us which documents are actually relevant to which topics.
# We use sep='\s+' because TREC files separate columns with spaces or tabs, not commas.
print("Loading Qrels...")
qrels_cols = ['Topic_ID', 'Iteration', 'Doc_ID', 'Relevance']
# UPDATE THIS PATH to where you saved the qrels file in your Drive
qrels_path = '/content/drive/MyDrive/IRWS_G12/set12/qrels.trec8.adhoc'
qrels_df = pd.read_csv(qrels_path, sep=r'\s+', names=qrels_cols)


# --- STEP 3: SYSTEM RUN LOADING ---
print("Loading System Runs...")
# We load the 6th column as 'Trash_Column' because we know it has broken 'nan' data
run_cols = ['Topic_ID', 'Iteration', 'Doc_ID', 'Rank', 'Score', 'Trash_Column']
run_files_path = '/content/drive/MyDrive/IRWS_G12/set12/System_Runs/*'

all_runs_list = []
for file in glob.glob(run_files_path):
    df = pd.read_csv(file, sep=r'\s+', names=run_cols)

    # Extract the system name directly from the filename
    # Example: 'path/to/input.Flab8as' -> 'input.Flab8as' -> 'Flab8as'
    file_name = os.path.basename(file)
    actual_system_name = file_name.split('.')[-1]

    # Create a brand new, clean System_Name column
    df['System_Name'] = actual_system_name

    all_runs_list.append(df)

full_runs_df = pd.concat(all_runs_list, ignore_index=True)

# Drop the broken 6th column to keep our data clean
full_runs_df.drop(columns=['Trash_Column'], inplace=True)

print("Data successfully loaded and cleaned using filenames!")


# --- STEP 4: PERFORM THE DEPTH AUDIT ---
# We group the data by System and Topic.
# The .count() function checks exactly how many Doc_IDs were returned for each grouping.
print("Running Depth Audit...")
depth_audit = full_runs_df.groupby(['System_Name', 'Topic_ID'])['Doc_ID'].count().reset_index()
depth_audit.rename(columns={'Doc_ID': 'Document_Count'}, inplace=True)

# Filter the audit to only show us the groupings that missed the 1000-document mark
anomalies = depth_audit[depth_audit['Document_Count'] < 1000]

print("\n--- AUDIT RESULTS ---")
print(f"Total system/topic combinations checked: {len(depth_audit)}")
print(f"Total anomalies found (< 1000 docs): {len(anomalies)}")
if len(anomalies) > 0:
    print("\nThese systems/topics need depth standardization:")
    print(anomalies)
else:
    print("\nAll systems successfully retrieved 1000 documents per topic!")


# --- STEP 5: THE RELEVANCE BASELINE CHECK ---
# Your team needs to know the total number of relevant documents per topic to calculate Recall and MAP.
# Want to know how many relevant documents actually exist for each topic in the qrel fileAq
# We filter the Qrels to only look at rows where Relevance is greater than 0 (i.e., relevant).
print("\n--- QREL BASELINE (Relevant Docs Per Topic) ---")
relevant_docs = qrels_df[qrels_df['Relevance'] > 0]
qrel_audit = relevant_docs.groupby('Topic_ID')['Doc_ID'].count().reset_index()
qrel_audit.rename(columns={'Doc_ID': 'Total_Relevant_Docs'}, inplace=True)

# Print the first 10 topics as a sanity check
print(qrel_audit.head(10))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading Qrels...
Loading System Runs...
Data successfully loaded and cleaned using filenames!
Running Depth Audit...

--- AUDIT RESULTS ---
Total system/topic combinations checked: 750
Total anomalies found (< 1000 docs): 5

These systems/topics need depth standardization:
    System_Name  Topic_ID  Document_Count
52      Flab8as       403              65
102     GE8MTD2       403              85
123     GE8MTD2       424             176
133     GE8MTD2       434             999
146     GE8MTD2       447             999

--- QREL BASELINE (Relevant Docs Per Topic) ---
   Topic_ID  Total_Relevant_Docs
0       401                  300
1       402                   80
2       403                   21
3       404                  142
4       405                   38
5       406                   13
6       407                   68
7       408                  118